# 06 · Skor konsep dari embedding (tanpa PCA, topik, atau classifier)
Kemiripan kosinus artikel dengan 6 kalimat jangkar → rata-rata harian → Ridge. Tidak ada yang dilatih di tahap teks.
Evaluasi **walk-forward** (mulai hari ke-60, refit tiap 5 hari, aturan alpha: CV minimum) di seluruh jendela 141 hari.
Pembanding utama: **rata-rata expanding** (bukan hanya nol), karena di notebook 04 selisih dari nol hanyalah rata-rata.
Didaftarkan sebelum dijalankan: A = harga; B = A + 6 jangkar; C = B + `sent_mean, gpr_mean, news_count, nov`.

In [1]:
import json
import numpy as np, pandas as pd
from utils import *

A = pd.read_parquet(PROC / 'articles.parquet')
EB = np.load(CACHE / 'emb_body.npy'); AN = np.load(CACHE / 'emb_anchors.npy')
an = json.load(open(CACHE / 'anchors.json'))
assert an['emb_model'] == json.load(open(CACHE / 'meta.json')).get('emb_model', an['emb_model']), 'model jangkar beda dengan model artikel'

df, price = load(with_pup=False)
ok = A.aligned_trading_date.isin(df.index).values
S = pd.DataFrame(EB[ok] @ AN.T, columns=an['names']).groupby(A.aligned_trading_date[ok].values).mean()
df[an['names']] = S.reindex(df.index).ffill().fillna(0)
EXTRA = ['sent_mean', 'gpr_mean', 'news_count', 'nov']
specs = {'A': price, 'B': price + an['names'], 'C': price + an['names'] + EXTRA}
print(len(df), 'hari |', df[an['names']].describe().loc[['mean', 'std']].round(3).to_string())

141 hari |       perang  sanksi  minyak  nuklir  diplomasi  ekonomi_as
mean   0.403   0.355   0.365   0.418      0.393       0.324
std    0.018   0.017   0.020   0.018      0.019       0.014


In [2]:
P = walk_forward(df, specs)
y = df.target.loc[P.index].values
yk = df.ret_lag0.loc[P.index].values
rows = {n: evaluate(y, P[n].values, df.target.std()) for n in specs}
rows |= {'mean': evaluate(y, P['mean'].values), 'zero': evaluate(y, 0 * y), 'kemarin': evaluate(y, yk)}
res = pd.DataFrame(rows).T
res['DM_vs_A_p'] = [dm_test(y, P['A'].values, P[n].values)[1] if n in ('B', 'C') else np.nan for n in res.index]
res['DM_vs_mean_p'] = [dm_test(y, P['mean'].values, P[n].values)[1] if n in specs else np.nan for n in res.index]
print(f'OOS n={len(P)} hari'); print(res.round(5).to_string())
for n in ('B', 'C'):
    lo, hi = rmse_diff_ci(y, P[n].values, P['A'].values)
    print(f'RMSE({n}) - RMSE(A) = {res.RMSE[n] - res.RMSE["A"]:+.6f}  CI95 [{lo:+.6f}, {hi:+.6f}]  (negatif = teks membantu)')

OOS n=81 hari
             RMSE       MAE    DirAcc collapsed  DM_vs_A_p  DM_vs_mean_p
A        0.003523  0.002658  0.530864      True        NaN       0.50931
B        0.003624  0.002726  0.506173     False    0.05981       0.06998
C        0.003603  0.002686  0.530864     False    0.09611       0.13312
mean     0.003515  0.002664  0.518519       NaN        NaN           NaN
zero     0.003504   0.00266  0.037037       NaN        NaN           NaN
kemarin  0.004814  0.003998  0.444444       NaN        NaN           NaN
RMSE(B) - RMSE(A) = +0.000100  CI95 [-0.000001, +0.000224]  (negatif = teks membantu)
RMSE(C) - RMSE(A) = +0.000079  CI95 [+0.000002, +0.000185]  (negatif = teks membantu)


## Kontrol negatif: acak urutan hari fitur teks (seluruh jendela), ulangi walk-forward

In [3]:
txt = an['names'] + EXTRA
neg = []
for s in range(20):
    sh = df.copy(); sh[txt] = sh[txt].sample(frac=1, random_state=s).values
    p = walk_forward(sh, {'C': specs['C']})['C'].values
    neg.append(np.sqrt(np.mean((y - p) ** 2)))
print(f'C asli {res.RMSE["C"]:.6f} | C acak {np.mean(neg):.6f} ± {np.std(neg):.6f} (min {np.min(neg):.6f}) | A {res.RMSE["A"]:.6f}')
print('Asli harus jelas di bawah distribusi acak. Kalau tidak, selisihnya bukan dari isi berita.')

C asli 0.003603 | C acak 0.003533 ± 0.000037 (min 0.003515) | A 0.003523
Asli harus jelas di bawah distribusi acak. Kalau tidak, selisihnya bukan dari isi berita.


In [4]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
Z = StandardScaler().fit_transform(df[specs['C']]); m = Ridge(alpha=100).fit(Z, df.target)   # alpha ilustratif, hanya untuk melihat tanda
print(pd.Series(m.coef_, specs['C']).sort_values().round(5).to_string(), '\n(ilustrasi saja, di-fit pada seluruh jendela, bukan hasil OOS)')

ekonomi_as       -0.00018
ret_lag1         -0.00014
ret_lag4         -0.00012
ret_lag3         -0.00010
nuklir           -0.00006
ret_roll_std5    -0.00005
ret_roll_mean5   -0.00004
perang           -0.00002
news_count        0.00000
nov               0.00001
sent_mean         0.00003
sanksi            0.00005
ret_lag0          0.00010
diplomasi         0.00011
minyak            0.00012
ret_lag2          0.00018
gpr_mean          0.00021 
(ilustrasi saja, di-fit pada seluruh jendela, bukan hasil OOS)
